# NB24 — Five-Factor Profit Decomposition for 73721
### *How the floor, the spread, and the structure come together*

**Procedure:** 73721 — knee MRI, no contrast. **Sample:** the five DFW hospitals.

| Hospital | CCN | County | Ownership | County safety-net |
|---|---|---|---|---|
| Baylor | 450021 | Dallas | Nonprofit | Parkland |
| Methodist | 450051 | Dallas | Nonprofit | Parkland |
| Parkland | 450015 | Dallas | **Public** | *is* the safety net |
| MCA (HCA) | 670103 | Tarrant | **For-profit** | JPS |
| THR Plano | 450771 | Collin | Nonprofit | **none** |

**Thesis (Decision 43):** 5 factors, two tiers.
Floor = *equipment · overhead · labor* (~constant, ≈ Medicare $244).
Spread = *profit margin · insurance negotiation* (the only movers).
**Cost explains the floor; market power explains the ceiling; the gap is the market failure.**

**Mold:** Step 0 → 1 → 2 → 3 → R → S → V → Findings → Integrity.
**Contract:** read-only lens; emits `outputs/nb24_*.json`; mutates no parent (like NB23).

> This is an **empty template** — the markdown says what each code cell must do; the code cells are yours to write.

## How to read / fill this notebook
Three honesty tiers to enforce in code:
- **MEASURED** — real transparency-file values (commercial dollar rates). Gate behind `USE_SYNTHETIC == False`.
- **MODELLED** — parametric estimate with a band (the cost-stack floor). Never a point measurement.
- **PROXY / PENDING** — profit margin. Keep behind `MARGIN_MEASURED = False`; never let a proxy enter measured tables.

Rule: if a step can only be done against the live repo, `raise NotImplementedError(...)` with a precise TODO rather than fabricate a number. **A blank is honest; a synthetic number is not.**

## Step 0 — Environment, config & source registry
Code cell(s) below should:
- import pandas/numpy/matplotlib; resolve `REPO / SRC / OUT / RAW` robustly.
- define the **control panel**: `TARGET_CODE="73721"`, `USE_SYNTHETIC`, `COMMERCIAL_DEF` (strict/private_inclusive/plus_aca), `MEDICARE_ALLOWED_73721` (≈244, TODO confirm), `R1_DOMINANCE_GATE=0.50`, `MA_CUES`, `MA_FALSE_FRIENDS=["texas advantage"]`, `MARGIN_MEASURED=False`.
- build the **CCN↔dbfile registry** + county/ownership/safety-net map + `STRUCTURAL_PREDICTION`.
- load `outputs/nb23_cleanup.json` and **assert `CLEANUP_VERIFIED`** (skip in synthetic mode).

In [ ]:
# Step 0a — imports + repo path resolver
from __future__ import annotations
import os, sys, json, math, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

def _find_repo_root(markers=("src", "outputs", "queries.py")) -> Path:
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand/"src"/"queries.py").exists() or (cand/"queries.py").exists():
            return cand
    return here

REPO = _find_repo_root(); SRC = REPO/"src"; OUT = REPO/"outputs"; RAW = REPO/"data"/"raw"
OUT.mkdir(exist_ok=True, parents=True)
print("REPO:", REPO)
print("SRC :", "OK" if (SRC/"queries.py").exists() else "MISSING (TODO fix path)")
print("RAW :", "OK" if RAW.exists() else "MISSING (TODO point at DuckDB dir)")

In [ ]:
# Step 0b — control panel (all knobs live here)
TARGET_CODE   = "73721"
USE_SYNTHETIC = True          
COMMERCIAL_DEF = "strict"     
MEDICARE_ALLOWED_73721 = 244.00   
R1_DOMINANCE_GATE = 0.50         
MA_CUES = ["medicare advantage","managed medicare","medicare complete","dual complete",
           "dual special","mapd","part c","blue advantage"]   # Decision 88
MA_FALSE_FRIENDS = ["texas advantage"]
MARGIN_MEASURED = False          
print(f"TARGET={TARGET_CODE} USE_SYNTHETIC={USE_SYNTHETIC} COMMERCIAL_DEF={COMMERCIAL_DEF} MARGIN_MEASURED={MARGIN_MEASURED}")

In [ ]:
# Step 0c — CCN<->dbfile registry + structural map
HOSPITALS = pd.DataFrame([
    ("450021","Baylor","Baylor University Medical Center","baylor","Dallas","nonprofit","Parkland"),
    ("450051","Methodist","Methodist Dallas Medical Center","methodist","Dallas","nonprofit","Parkland"),
    ("450015","Parkland","Parkland Health","parkland","Dallas","public","self"),
    ("670103","MCA","Medical City (MCA / HCA)","alliance","Tarrant","for-profit","JPS"),
    ("450771","THR_Plano","Texas Health Presbyterian Plano","plano","Collin","nonprofit","none"),
], columns=["ccn","short","name","dbfile_keyword","county","ownership","safety_net"]).set_index("ccn")

STRUCTURAL_PREDICTION = {
    "450021":"mid-high; burden absorbed by Parkland -> markup=negotiation, not subsidy",
    "450051":"mid; burden absorbed by Parkland",
    "450015":"n/a; public safety-net, %-of-charges, no clean commercial fee line",
    "670103":"wide payer spread via chargemaster (for-profit); negotiated commercial not necessarily high",
    "450771":"PREDICTED HIGHEST; no county safety-net -> uncompensated care cost-shifts into commercial",
}
HOSPITALS

In [ ]:
# Step 0d — load & verify nb23_cleanup.json (gate on CLEANUP_VERIFIED)
CLEANUP_PATH = OUT/"nb23_cleanup.json"
def load_cleanup_manifest(path: Path = CLEANUP_PATH) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run NB23 first (emits the verified manifest).")
    m = json.loads(path.read_text())
    if not m.get("CLEANUP_VERIFIED", False):
        raise AssertionError("nb23_cleanup.json present but CLEANUP_VERIFIED != True. Do not proceed.")
    return m

CLEANUP = None
if not USE_SYNTHETIC:
    CLEANUP = load_cleanup_manifest()
    print("Loaded verified cleanup manifest.")
else:
    print("USE_SYNTHETIC=True -> manifest load skipped; no measured claims made.")

## Step 1 — The floor: equipment + overhead + labor  *(MODELLED)*
Show the 73721 production cost lands in the low hundreds ≈ Medicare, so the floor is shared and everything above it is the two spread factors.
Code cell(s) below should:
- build a parametric `CostStack` (capital amortized: magnet $150K–$3M + RF room; opex: service/helium/power ~$800K–1.5M/yr; labor: tech + radiologist read; supplies $0), returning a low/mid/high **floor band**.
- compare the band to `MEDICARE_ALLOWED_73721`.
- add a **utilization sensitivity** (floor vs scans/year) to show the floor is driven by volume, not sticker price.

In [ ]:
# Step 1a — parametric cost-stack model -> floor band vs Medicare
@dataclass
class CostStack:
    magnet_cost:tuple=(150_000,3_000_000); room_cost:tuple=(200_000,500_000); useful_life_yrs:float=10.0
    service_contract:tuple=(100_000,300_000); helium:tuple=(15_000,60_000); power:tuple=(20_000,80_000)
    other_facility:tuple=(50_000,150_000); scans_per_year:tuple=(2_000,4_000)
    tech_minutes:float=45.0; tech_hourly:float=45.0; rad_read_fee:tuple=(35.0,60.0); supplies_per_scan:float=0.0
    def _mid(self,t): return sum(t)/2 if isinstance(t,tuple) else t
    def per_scan(self, level="mid"):
        pick=(lambda t: t[0] if level=="low" else t[1] if level=="high" else self._mid(t))
        cap=(pick(self.magnet_cost)+pick(self.room_cost))/self.useful_life_yrs
        opex=sum(pick(x) for x in (self.service_contract,self.helium,self.power,self.other_facility))
        scans=pick(self.scans_per_year); fixed=(cap+opex)/scans
        labor=(self.tech_minutes/60.0)*self.tech_hourly + pick(self.rad_read_fee) + self.supplies_per_scan
        return {"equipment+overhead_per_scan":fixed,"labor_per_scan":labor,"floor_per_scan":fixed+labor}

STACK = CostStack()
floor_band = {l: STACK.per_scan(l)["floor_per_scan"] for l in ("low","mid","high")}
print("Modelled floor per knee MRI (no contrast):")
for l,v in floor_band.items(): print(f"  {l:>4}: ${v:,.0f}")
print(f"Medicare allowed anchor: ${MEDICARE_ALLOWED_73721:,.2f}  ->  floor overlaps Medicare (cost-recovery).")

In [ ]:
# Step 1b — floor sensitivity to utilization (scans/year)
def floor_vs_utilization(grid=(1000,1500,2000,3000,4000,6000)):
    return pd.DataFrame([{"scans_per_year":s,
        "floor_per_scan":CostStack(scans_per_year=(s,s)).per_scan("mid")["floor_per_scan"]} for s in grid])
floor_vs_utilization()

## Step 2 — Spread lever #1: insurance negotiation  *(MEASURED)*
The finalized commercial dollar spread *is* the signature of negotiation power.
Day-31 ground truth to reproduce: Baylor med **1,868** (n11) › Methodist **1,593** (n16) › THR **1,263** (n12) › MCA **907** (n14) › Parkland **NaN** (n1).
Code cell(s) below should:
- live loader mirroring NB13/16/23: `queries.py::query_procedure_rates_agg` per DuckDB → `add_lob_v4` → **MA-recovery guard** → **cleanup rules R1/R2/R3** → `COMMERCIAL_DEF` mask. (Provide a clearly-fake synthetic stand-in for `USE_SYNTHETIC=True`.)
- per-hospital **spread table** (rows, n_valid, min, max, median, `markup_x_floor`); keep Parkland NaN, not 0.
- a **regression check** asserting the medians/row-counts equal the Day-31 ground truth on real data.

In [ ]:
# Step 2a — live commercial loader (+ synthetic stand-in) 
def _resolve_dbfile(keyword: str) -> Path:
    matches = sorted(RAW.glob(f"*{keyword}*.duckdb")) if RAW.exists() else []
    if not matches:
        raise NotImplementedError(f"No DuckDB matched '{keyword}' under {RAW}. Fix RAW / naming map.")
    if len(matches) > 1:
        raise ValueError(f"Ambiguous keyword '{keyword}' matched {len(matches)} DuckDBs: "
                         f"{[m.name for m in matches]}. Make the keyword unique.")
    return matches[0]

def _load_real_commercial(code: str = TARGET_CODE) -> pd.DataFrame:
    """Pull rows from the five DuckDBs, classify LOB, apply NB23 rules; return long frame."""
    if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
    try:
        import duckdb
        from queries import query_procedure_rates_agg      # noqa
        from queries import add_lob_v4                          # noqa  TODO confirm module path
    except Exception as e:
        raise NotImplementedError(
            f"Wire live imports before USE_SYNTHETIC=False: {e!r} "
            "(expected src/queries.py::query_procedure_rates_agg + add_lob_v4).")
    frames=[]
    for ccn,row in HOSPITALS.iterrows():
        con = duckdb.connect(str(_resolve_dbfile(row["dbfile_keyword"])), read_only=True)
        try:
            df = query_procedure_rates_agg(con, code)
        finally:
            con.close()
        df["ccn"], df["short"] = ccn, row["short"]; frames.append(df)
    raw = pd.concat(frames, ignore_index=True)
    raw = add_lob_v4(raw)
    raw = apply_ma_recovery_guard(raw)      # defined in 2b
    raw = apply_cleanup_rules(raw)          # defined in 2b
    raw["is_commercial"] = commercial_mask(raw, COMMERCIAL_DEF)
    return raw

def _synthetic_commercial() -> pd.DataFrame:
    rows=[]
    for ccn,r in HOSPITALS.iterrows():
        n = 3 if ccn!="450015" else 1
        for _ in range(n):
            rows.append({"ccn":ccn,"short":r["short"],"payer":"SYNTH","plan_name":"SYNTH",
                         "dollar_rate":(np.nan if ccn=="450015" else 1.0),"pct_rate":np.nan,
                         "methodology_normalized":"case rate","lob_v4":"commercial","is_commercial":True})
    df=pd.DataFrame(rows); df.attrs["SYNTHETIC"]=True; return df

def get_commercial_frame() -> pd.DataFrame:
    if USE_SYNTHETIC:
        print("[SYNTHETIC DATA] shape-only stand-in; no measured claims."); return _synthetic_commercial()
    return _load_real_commercial()

In [ ]:
# Step 2b — cleanup rule application: MA-recovery guard + R1/R2/R3 + commercial mask
def apply_ma_recovery_guard(df: pd.DataFrame) -> pd.DataFrame:
    """Strip Medicare-Advantage rows add_lob_v4 misfiled as commercial (Decision 88)."""
    pn = df.get("plan_name", pd.Series("", index=df.index)).fillna("")
    py = df.get("payer", pd.Series("", index=df.index)).fillna("")
    text = (pn + " " + py).str.lower()
    is_ma = text.apply(lambda s: any(c in s for c in MA_CUES) and not any(f in s for f in MA_FALSE_FRIENDS))
    if "lob_v4" in df.columns:
        df.loc[is_ma & df["lob_v4"].eq("commercial"), "lob_v4"] = "medicare_advantage"
    df["_ma_recovered"] = is_ma
    return df

def commercial_mask(df: pd.DataFrame, definition: str) -> pd.Series:
    base = df["lob_v4"].eq("commercial")
    if definition=="strict": return base
    if definition=="private_inclusive": return base | df["lob_v4"].isin(["private","self_insured","other_private"])
    if definition=="plus_aca": return base | df["lob_v4"].isin(["private","self_insured","other_private","aca","marketplace"])
    raise ValueError(f"unknown COMMERCIAL_DEF: {definition}")

def apply_cleanup_rules(df: pd.DataFrame) -> pd.DataFrame:
    """R1 per-hospital placeholder floor (dominance-gated), R2 AETNA transplant, R3 %-charges."""
    if CLEANUP is None:
        raise NotImplementedError("CLEANUP manifest not loaded (USE_SYNTHETIC=True).")
    keep = pd.Series(True, index=df.index)
    keep &= ~df["methodology_normalized"].str.lower().eq("percent of total billed charges")   # R3
    txt = (df.get("plan_name","").fillna("") + " " + df.get("payer","").fillna("")).str.lower()
    keep &= ~(txt.str.contains("aetna") & txt.str.contains("transplant"))                       # R2
    for ccn,g in df.groupby("ccn"):                                                             # R1
        vc = g["dollar_rate"].round(2).value_counts(normalize=True, dropna=True)
        if len(vc) and vc.iloc[0] >= R1_DOMINANCE_GATE:
            keep &= ~(df["ccn"].eq(ccn) & df["dollar_rate"].round(2).eq(vc.index[0]))
    return df[keep].copy()

In [ ]:
# Step 2c — per-hospital commercial spread table + markup over floor
commercial = get_commercial_frame()

def commercial_spread_table(df: pd.DataFrame) -> pd.DataFrame:
    c = df[df["is_commercial"]].copy()
    agg = (c.groupby(["ccn","short"])["dollar_rate"]
             .agg(rows="count", n_valid=lambda s:int(s.notna().sum()), min="min", max="max", median="median")
             .reset_index())
    agg["markup_x_floor"] = agg["median"]/MEDICARE_ALLOWED_73721
    agg.loc[agg["n_valid"].eq(0), ["median","markup_x_floor"]] = np.nan   # Parkland: NaN, not 0
    return agg.sort_values("median", ascending=False, na_position="last")

spread = commercial_spread_table(commercial)
spread

In [ ]:
# Step 2d — regression check vs Day-31 ground truth (medians + row counts)
DAY31_MEDIANS = {"450021":1868.00,"450051":1592.99,"450771":1262.69,"670103":906.53,"450015":np.nan}
DAY31_ROWS    = {"450021":11,"450051":16,"450771":12,"670103":14,"450015":1}

def check_against_day31(spread_df, tol=0.01):
    if USE_SYNTHETIC:
        print("USE_SYNTHETIC=True -> regression check skipped."); return
    ok=True
    for _,r in spread_df.iterrows():
        exp=DAY31_MEDIANS[r["ccn"]]; got=r["median"]
        match=(np.isnan(exp) and r["n_valid"]==0) or (not np.isnan(exp) and abs(got-exp)<=max(1.0,tol*exp))
        ok &= bool(match)
        print(f"  {r['short']:<10} got={got!s:>9} exp={exp!s:>9} rows {r['rows']}/{DAY31_ROWS[r['ccn']]} {'OK' if match else 'MISMATCH'}")
    assert ok, "Step-2 does not reproduce Day-31 ground truth."
    print("Reproduces Day-31 ground truth.")

check_against_day31(spread)

## Step 3 — Spread lever #2: profit margin *(HELD PENDING)* + structural overlay
Margin is the factor that disambiguates *profit* from *burden*. It is hospital-wide, not in the transparency files, and blocked on the HCRIS pull (Decision 81).
Code cell(s) below should:
- an HCRIS margin loader that **`raise NotImplementedError`** until the pull lands; a markup **proxy** that is tagged `estimate` and asserts `MARGIN_MEASURED is False`.
- the **county × ownership natural experiment**: join `STRUCTURAL_PREDICTION` to the Step-2 spread; compute observed rank; surface the THR-Plano predicted-highest-vs-observed-midpack check.

In [ ]:
# Step 3a — profit margin: HCRIS loader stub (raises) + estimate-only proxy
def pull_hcris_margin(ccn: str) -> float:
    raise NotImplementedError("HCRIS margin pull open (Day-81). One download clears NB19/20/21/22 + this factor.")

def margin_proxy_markup(spread_df: pd.DataFrame) -> pd.DataFrame:
    """PROXY ONLY. Markup-over-Medicare as pricing-power stand-in. NOT profit. Tagged estimate."""
    p = spread_df[["ccn","short","median","markup_x_floor"]].copy()
    p["margin_value"]  = np.nan
    p["margin_source"] = "PENDING_HCRIS"
    p["proxy_markup_x_floor"] = p["markup_x_floor"]
    p["confidence_tier"] = "proxy/estimate"
    assert not MARGIN_MEASURED, "MARGIN_MEASURED must stay False until HCRIS lands."
    return p

margin = margin_proxy_markup(spread)
margin

In [ ]:
# Step 3b — county x ownership overlay: predicted vs observed rank
def structural_overlay(spread_df: pd.DataFrame) -> pd.DataFrame:
    hp = HOSPITALS.reset_index()[["ccn","county","ownership","safety_net"]]
    s = spread_df.merge(hp, on="ccn", how="left")
    s["structural_prediction"] = s["ccn"].map(STRUCTURAL_PREDICTION)
    ranked = s.dropna(subset=["median"]).sort_values("median", ascending=False).reset_index(drop=True)
    ranked["observed_rank"] = ranked.index + 1
    s = s.merge(ranked[["ccn","observed_rank"]], on="ccn", how="left")
    return s.sort_values("median", ascending=False, na_position="last")

overlay = structural_overlay(spread)
overlay[["short","county","ownership","safety_net","median","markup_x_floor","observed_rank","structural_prediction"]]


## R — Results: the five factors assembled
Code cell(s) below should:
- assemble the master table: shared `floor`, `medicare_allowed`, `commercial_median` (MEASURED), `profit_margin` (blank/PENDING).
- draw the **price ladder** (log scale): floor → Medicare → commercial median → cash/chargemaster (MCA cash ≈ $8,364; national cash avg ≈ $1,325).
- emit `outputs/nb24_five_factor.json` (+ `nb24_price_ladder.png`), gated so synthetic runs don't write sentinels.

In [ ]:
# R-a — assemble the five-factor master table
def assemble_five_factor(spread_df: pd.DataFrame) -> pd.DataFrame:
    hp = HOSPITALS.reset_index()[["ccn","county","ownership"]]
    out = spread_df.merge(hp, on="ccn", how="left")
    out["floor_shared"]     = MEDICARE_ALLOWED_73721   # equipment+overhead+labor (MODELLED, shared)
    out["medicare_allowed"] = MEDICARE_ALLOWED_73721
    out["commercial_median"]= out["median"]            # MEASURED (negotiation)
    out["profit_margin"]    = np.nan                   # PENDING (HCRIS)
    out["margin_status"]    = "PENDING_HCRIS"
    cols=["ccn","short","county","ownership","floor_shared","medicare_allowed",
          "commercial_median","markup_x_floor","profit_margin","margin_status","rows","n_valid"]
    return out[cols].sort_values("commercial_median", ascending=False, na_position="last")

five_factor = assemble_five_factor(spread)
five_factor

In [ ]:
# R-b — price ladder figure (log scale)
CASH_ANCHORS = {"670103":8364.0}     # MCA cash = chargemaster (Day-20); others TODO
NATIONAL_CASH_AVG = 1325.0

def plot_price_ladder(five_df, save_to=OUT/"nb24_price_ladder.png"):
    fig, ax = plt.subplots(figsize=(10,6))
    order = five_df.dropna(subset=["commercial_median"]).sort_values("commercial_median")
    y = np.arange(len(order))
    ax.axvline(MEDICARE_ALLOWED_73721, ls="--", lw=1, label=f"Medicare ${MEDICARE_ALLOWED_73721:,.0f} (≈floor)")
    ax.axvline(NATIONAL_CASH_AVG, ls=":", lw=1, color="grey", label=f"nat cash avg ${NATIONAL_CASH_AVG:,.0f}")
    ax.scatter(order["commercial_median"], y, s=70, label="commercial median (measured)", zorder=3)
    for i,(_,r) in enumerate(order.iterrows()):
        cash = CASH_ANCHORS.get(r["ccn"])
        if cash:
            ax.scatter([cash],[i], marker="x", color="firebrick", zorder=3)
            ax.annotate(f"cash ${cash:,.0f}", (cash,i), xytext=(6,0), textcoords="offset points",
                        va="center", fontsize=8, color="firebrick")
    ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels([r["short"] for _,r in order.iterrows()])
    ax.set_xlabel("price for 73721 (USD, log scale)")
    ax.set_title("73721 price ladder — shared floor vs negotiated commercial spread")
    ax.legend(loc="lower right", fontsize=8); fig.tight_layout()
    if not USE_SYNTHETIC: fig.savefig(save_to, dpi=140); print("saved", save_to)
    else: print("[SYNTHETIC DATA] figure not saved (sentinels).")
    plt.close(fig)

plot_price_ladder(five_factor)

In [ ]:
# R-c — emit outputs/nb24_five_factor.json (gate on USE_SYNTHETIC)
def emit_manifest(five_df, path=OUT/"nb24_five_factor.json"):
    payload = {"notebook":"NB24","target_code":TARGET_CODE,"commercial_def":COMMERCIAL_DEF,
               "medicare_allowed":MEDICARE_ALLOWED_73721,"floor_model":floor_band,"margin_measured":MARGIN_MEASURED,
               "per_hospital":json.loads(five_df.to_json(orient="records")),
               "provenance":{"cleanup_manifest":str(CLEANUP_PATH),
                             "cleanup_verified":(None if CLEANUP is None else CLEANUP.get("CLEANUP_VERIFIED")),
                             "synthetic":USE_SYNTHETIC}}
    if USE_SYNTHETIC: print("[SYNTHETIC DATA] manifest not emitted."); return payload
    path.write_text(json.dumps(payload, indent=2)); print("emitted", path); return payload

_manifest = emit_manifest(five_factor)

## S — Sensitivity & knobs
Code cell(s) below should:
- **COMMERCIAL_DEF sweep** (strict 54 / private_inclusive 66 / plus_aca 72): does the Baylor>Methodist>THR>MCA order survive broadening?
- **R1 dominance-gate sweep** (0.30–0.70): which per-hospital placeholder floors drop — re-derive **per code** (232.47 is Methodist-and-73721-specific).
- **floor robustness**: floor band vs Medicare and vs the cheapest commercial median (claim = floor ≈ Medicare and ≪ spread).

In [ ]:
# S-a — COMMERCIAL_DEF sweep
def commercial_def_sweep(df, defs=("strict","private_inclusive","plus_aca")):
    if USE_SYNTHETIC: print("USE_SYNTHETIC=True -> shape-only sweep.")
    out={}
    for d in defs:
        m = commercial_mask(df,d) if not USE_SYNTHETIC else df["is_commercial"]
        out[d] = df[m].groupby("short")["dollar_rate"].median().sort_values(ascending=False)
    return pd.DataFrame(out)

commercial_def_sweep(commercial)

In [ ]:
# S-b — R1 dominance-gate sweep (per code)

In [ ]:
# S-c — floor robustness across the cost-stack band

## V — Verification locks
Code cell below should assert, then set `DECOMP_VERIFIED`:
1. no synthetic leakage into measured claims; 2. **zero MA rows** survive in commercial; 3. Parkland has **no usable rate** (NaN, not 0); 4. modelled floor **< every** commercial median; 5. `MARGIN_MEASURED is False` and margin column all-NaN; 6. consumed cleanup manifest is `CLEANUP_VERIFIED` on real runs.

In [ ]:
# V — verification battery -> DECOMP_VERIFIED

## Findings
*(Write after the real-data run.)* Expected shape from Days 20–31:
1. Floor is one shared story ≈ Medicare (utilization-driven).
2. Negotiation fans out ~3.7×–7.7× the floor (Baylor priciest → MCA cheapest).
3. For-profit tension: MCA games the chargemaster hardest yet has the *lowest* negotiated commercial rate → margin needed to disambiguate.
4. Structural prediction is a partial hit: THR Plano predicted highest, lands mid-pack — report honestly.
5. Parkland's empty commercial side is a headline (market opacity), not a gap.
6. Profit margin is the missing overlay of known shape — HCRIS fills it.

## Integrity — provenance, vintage, caveats, decisions
Code cell below should record: inputs (DuckDBs via queries.py; add_lob_v4 + MA guard; nb23_cleanup.json; Medicare anchor TODO; cost stack MODELLED), confidence tiers per factor, vintage TODO (per-file `last_updated`), the caveats list, and proposed Decisions 91–94.

In [ ]:
# Integrity — provenance / caveats / proposed decisions dict

### Open items / handoff
- Wire the live loader (`add_lob_v4` path, `RAW` DuckDB dir + filename→hospital map); flip `USE_SYNTHETIC=False`; pass the Step-2 regression check.
- Confirm the exact MPFS/Addendum-B Medicare allowed for 73721 (replace the ~244 placeholder).
- Fill the margin factor only after the HCRIS pull lands (Decision 81) with a per-hospital vs per-procedure guard.
- Verify the CCN↔dbfile crosswalk against CMS provider IDs.
- Read each file's `last_updated` before any public/app output.
- Do not port upstream without a separate explicit step — NB24 is a lens.